# Variables

For the data imputation process we will focus on the following variables:
- PM2.5 (μg/m³): material particulado menor a 2.5 micrómetros.
- NO₂ (ppb): dióxido de nitrógeno.
- CO (ppm): monóxido de carbono.
- NO (ppb): monóxido de nitrógeno.
- NOx (ppb): suma de NO y NO₂.

And the following stations:
- NO y CE - Monterrey
- NE - San Nicolás de los Garza
- SE - Guadalupe
- NE2 - Apodaca
- SO - Santa Catarina
- NO2 - García

In [3]:
# Imports and setup
import os
import numpy as np
import pandas as pd


# Load dataset and isolate selected variables/stations

This section loads the unified dataset and isolates only the variables and stations listed above. It follows the same conventions as the other notebooks (parse_dates, replace -9999 with NaN).


In [4]:
# Define target pollutants from markdown (normalize names to match dataset columns)
pollutants_markdown = ["PM2.5", "NO₂", "CO", "NO", "NOx"]
normalize_map = {"NO₂": "NO2", "NOx": "NOX"}
pollutants = [normalize_map.get(p, p) for p in pollutants_markdown]

# Define target stations and corresponding station codes
stations_markdown = [
    "NO y CE - Monterrey",
    "NE - San Nicolás de los Garza",
    "SE - Guadalupe",
    "NE2 - Apodaca",
    "SO - Santa Catarina",
    "NO2 - García",
]

# Map station labels from markdown to station_code values used in the main dataset
station_codes_map = {
    "NO y CE - Monterrey": ["NO", "CE"],
    "NE - San Nicolás de los Garza": ["NE"],
    "SE - Guadalupe": ["SE"],
    "NE2 - Apodaca": ["NE2"],
    "SO - Santa Catarina": ["SO"],
    "NO2 - García": ["NO2"],
}

target_station_codes = sorted({code for name in stations_markdown for code in station_codes_map.get(name, [])})

# Load dataset (same settings as in exploration notebook)
data_path = os.path.join("..", "data", "processed", "main_dataframe.csv")
df = pd.read_csv(
    data_path,
    parse_dates=["date"],
    engine="pyarrow"
)

# Treat sentinel missing values
df.replace(-9999, np.nan, inplace=True)

# Validate and select pollutant columns that exist
available_pollutants = [c for c in pollutants if c in df.columns]
missing_pollutants = [c for c in pollutants if c not in df.columns]
if missing_pollutants:
    print(f"Warning: the following pollutant columns were not found and will be skipped: {missing_pollutants}")

# Filter by station codes and keep only the relevant columns
keep_cols = ["date", "station_code"] + available_pollutants
subset = df[df["station_code"].astype(str).isin(target_station_codes)][keep_cols].copy()

# Sort by date for readability
if "date" in subset.columns:
    subset.sort_values("date", inplace=True)

# Save isolated subset
output_dir = os.path.join("..", "data", "processed")
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "imputation_subset.csv")
subset.to_csv(output_file, index=False)

print(f"Saved isolated subset to: {output_file}")
print("Station codes included:", target_station_codes)
print("Columns:", keep_cols)
print("Rows:", len(subset))

# Quick preview
subset.head()


Saved isolated subset to: ..\data\processed\imputation_subset.csv
Station codes included: ['CE', 'NE', 'NE2', 'NO', 'NO2', 'SE', 'SO']
Columns: ['date', 'station_code', 'PM2.5', 'NO2', 'CO', 'NO', 'NOX']
Rows: 375587


,date,station_code,PM2.5,NO2,CO,NO,NOX
0,2020-01-01,SE,54.23,NaN,NaN,NaN,NaN
35073,2020-01-01,CE,60.91,NaN,NaN,NaN,NaN
52609,2020-01-01,NO,70.81,NaN,NaN,NaN,NaN
70146,2020-01-01,SO,59.26,NaN,NaN,NaN,NaN
87683,2020-01-01,NO2,86.93,NaN,NaN,NaN,NaN
